# Task 3: Vectorized Conv2D via im2col (NumPy only)

In [1]:
import numpy as np


In [2]:
def get_indices(X_shape, HF, WF, stride, pad):
    m, n_C, n_H, n_W = X_shape
    out_h = (n_H + 2*pad - HF)//stride + 1
    out_w = (n_W + 2*pad - WF)//stride + 1

    level1 = np.repeat(np.arange(HF), WF)
    level1 = np.tile(level1, n_C)
    everyLevels = stride * np.repeat(np.arange(out_h), out_w)
    i = level1.reshape(-1,1) + everyLevels.reshape(1,-1)

    slide1 = np.tile(np.arange(WF), HF)
    slide1 = np.tile(slide1, n_C)
    everySlides = stride * np.tile(np.arange(out_w), out_h)
    j = slide1.reshape(-1,1) + everySlides.reshape(1,-1)

    d = np.repeat(np.arange(n_C), HF*WF).reshape(-1,1)
    return i, j, d

def im2col(X, HF, WF, stride, pad):
    X_padded = np.pad(X, ((0,0),(0,0),(pad,pad),(pad,pad)), mode="constant")
    i, j, d = get_indices(X.shape, HF, WF, stride, pad)
    cols = X_padded[:, d, i, j]
    cols = cols.transpose(1,2,0).reshape(HF*WF*X.shape[1], -1)
    return cols


In [3]:
def conv2d_forward(X, W, b, stride=1, pad=0):
    m, n_C_prev, n_H_prev, n_W_prev = X.shape
    n_C, _, HF, WF = W.shape
    n_H = (n_H_prev + 2*pad - HF)//stride + 1
    n_W = (n_W_prev + 2*pad - WF)//stride + 1

    X_col = im2col(X, HF, WF, stride, pad)
    W_col = W.reshape(n_C, -1)

    out = W_col @ X_col + b.reshape(-1,1)
    out = out.reshape(n_C, n_H, n_W, m).transpose(3,0,1,2)
    return out


In [4]:
np.random.seed(0)
X = np.random.randn(2, 3, 8, 8)
W = np.random.randn(4, 3, 3, 3)
b = np.random.randn(4)

out = conv2d_forward(X, W, b, stride=1, pad=1)
print(out.shape)


(2, 4, 8, 8)
